# Modeldiffr on a free Colab GPU

Audit what changed between Qwen2.5 0.5B Instruct and its base model with the `lite` suite: about 1,000 items across knowledge, math, instruction following, refusals and calibration. Runtime: Runtime, Change runtime type, T4 GPU.

In [ ]:
!pip -q install "modeldiffr[hf,tasks,proof]>=0.3"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Plan

Prints the items per task, the smallest change each task can detect, and the expected run time on this GPU.

In [ ]:
BASE = "Qwen/Qwen2.5-0.5B-Instruct"
DERIVATIVE = "Qwen/Qwen2.5-0.5B"
!modeldiffr audit {BASE} {DERIVATIVE} --suite lite --dtype float16 --plan-only --estimate

## Audit

In [ ]:
!modeldiffr audit {BASE} {DERIVATIVE} --suite lite --dtype float16 --out /content/audit

## Verify

Recomputes every hash and metric from the report, rebuilds the suite from the pinned datasets, and checks the weights against the Hub.

In [ ]:
!modeldiffr verify /content/audit/report.json --suite-data hub --models

## Read the report

In [ ]:
import json
import pandas as pd
from IPython.display import HTML, display

report = json.load(open("/content/audit/report.json"))
print("audit id", report["audit_id"])
print("changed:", report["verdict"]["changed"] or "nothing")
cols = ["name", "base_mean", "derivative_mean", "delta", "ci_low", "ci_high", "p_value", "significant", "negligible"]
display(pd.DataFrame(report["metrics"])[cols].round(4))
display(HTML(open("/content/audit/report.html").read()))

In [ ]:
!cd /content && zip -qr audit.zip audit
from google.colab import files
files.download("/content/audit.zip")